## Setup and Imports

In [21]:
import os
import numpy as np
import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification
)
from torch.utils.data import Dataset
import pandas as pd
from tqdm import tqdm

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("Setup complete")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Setup complete
PyTorch version: 2.11.0.dev20260204+cu128
CUDA available: True


###  Define label space (entity types + BIO tagging)
 definition of the 13 GutBrainIE entity categories and expansions into BIO tags:
 - "O" for tokens outside any entity
 - "B-<label>" for the first token of an entity mention
 - "I-<label>" for continuation tokens
 Then we build `label2id` / `id2label` mappings so the model can train and decode labels.

 Finally we set:
 - the pretrained backbone (BioBERT)
 - the output directory where the fine-tuned model will be saved.

In [22]:
# Define entity labels
ENTITY_LABELS = [
    "anatomical location",
    "animal",
    "bacteria",
    "biomedical technique",
    "chemical",
    "DDF",
    "dietary supplement",
    "drug",
    "food",
    "gene",
    "human",
    "microbiome",
    "statistical technique"
]

# Create BIO tags for each entity label
label_list = ['O']  # Outside
for entity_label in ENTITY_LABELS:
    label_list.append(f'B-{entity_label}')  # Beginning
    label_list.append(f'I-{entity_label}')  # Inside

label2id = {k: v for v, k in enumerate(label_list)}
id2label = {v: k for v, k in enumerate(label_list)}

print(f"Total labels: {len(label_list)}")
print(f"\nFirst 10 labels: {label_list[:10]}")

# Model configuration
model_name = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract"
output_model_dir = "models/pubmedbert_ner_twopass_gold_silver_bronze"

print(f"\nModel: {model_name}")
print(f"Output directory: {output_model_dir}")

Total labels: 27

First 10 labels: ['O', 'B-anatomical location', 'I-anatomical location', 'B-animal', 'I-animal', 'B-bacteria', 'I-bacteria', 'B-biomedical technique', 'I-biomedical technique', 'B-chemical']

Model: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract
Output directory: models/pubmedbert_ner_twopass_gold_silver_bronze


## Data loading utilities (merge by priority + segment-level examples)
This cell defines helper functions to: (1) load JSON annotation files, (2) merge training sources with a priority policy (e.g., gold overrides silver), and (3) convert each article into two training examples (title and abstract) so that entity offsets remain consistent with the annotated `location`.

In [23]:
from pathlib import Path


def load_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

def load_ner_data_priority(train_paths_by_quality):
    """
    train_paths_by_quality: list of tuples (quality_name, path)
    Priority order is the given order (first wins).
    """
    merged = {}
    source = {}  # pmid -> quality

    for quality, path in train_paths_by_quality:
        if not path.exists():
            print(f"⚠️ Missing: {path}")
            continue

        data = load_json(path)
        print(f"Loaded {len(data)} docs from {path.name} ({quality})")

        for pmid, article in data.items():
            # first wins: gold > silver > bronze
            if pmid not in merged:
                merged[pmid] = article
                source[pmid] = quality

    print(f"✓ Merged unique PMIDs: {len(merged)} (priority kept: gold>silver>bronze)")
    return merged

def prepare_documents_for_ner(data):
    documents = []
    for pmid, article in data.items():
        meta = article.get("metadata", {})
        title_text = (meta.get("title") or "").strip()
        abstract_text = (meta.get("abstract") or "").strip()
        entities = article.get("entities", []) or []

        # title segment
        if title_text:
            title_entities = [e for e in entities if e.get("location") == "title"]
            documents.append({
                "pmid": str(pmid),
                "location": "title",
                "text": title_text,
                "entities": title_entities
            })

        # abstract segment
        if abstract_text:
            abstract_entities = [e for e in entities if e.get("location") == "abstract"]
            documents.append({
                "pmid": str(pmid),
                "location": "abstract",
                "text": abstract_text,
                "entities": abstract_entities
            })

    return documents
print("✓ Data loading functions defined")

✓ Data loading functions defined


## Load Training and Dev Data
loading of the training data (gold/platinum/silver) and development data from the provided dev split. Then it converts articles into per-segment examples (title + abstract), producing `train_documents` and `dev_documents`.

In [24]:
import json
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """
    Risale le cartelle finché trova una directory che contiene 'data'.
    Funziona anche se il notebook parte da src/ner/...
    """
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / "data").exists():
            return p
    raise FileNotFoundError(f"Non trovo la cartella 'data' risalendo da: {start}")

PROJECT_ROOT = find_repo_root(Path.cwd())
DATA_ROOT = PROJECT_ROOT / "data" / "GutBrainIE_Full_Collection_2026"
ANNOTATIONS_DIR = DATA_ROOT / "Annotations"

# --- Train files (2026) ---
train_files = [
    ANNOTATIONS_DIR / "Train" / "gold_quality"   / "json_format" / "train_gold.json",
    ANNOTATIONS_DIR / "Train" / "silver_quality" / "json_format" / "train_silver.json",
    ANNOTATIONS_DIR / "Train" / "bronze_quality" / "json_format" / "train_bronze.json"

]

# opzionale: aggiungi anche silver_2025 se vuoi
train_files_optional = [
    ANNOTATIONS_DIR / "Train" / "silver_quality" / "json_format" / "train_silver_2025.json"
]

# --- Dev file ---
dev_file = ANNOTATIONS_DIR / "Dev" / "json_format" / "dev.json"

print("Train files:")
for p in train_files + train_files_optional:
    print(" -", p, "| exists:", p.exists())
print("Dev file:", dev_file, "| exists:", dev_file.exists())

Train files:
 - C:\Users\super\Documents\UniPd\ATA\GutBrainIE\data\GutBrainIE_Full_Collection_2026\Annotations\Train\gold_quality\json_format\train_gold.json | exists: True
 - C:\Users\super\Documents\UniPd\ATA\GutBrainIE\data\GutBrainIE_Full_Collection_2026\Annotations\Train\silver_quality\json_format\train_silver.json | exists: True
 - C:\Users\super\Documents\UniPd\ATA\GutBrainIE\data\GutBrainIE_Full_Collection_2026\Annotations\Train\bronze_quality\json_format\train_bronze.json | exists: True
 - C:\Users\super\Documents\UniPd\ATA\GutBrainIE\data\GutBrainIE_Full_Collection_2026\Annotations\Train\silver_quality\json_format\train_silver_2025.json | exists: True
Dev file: C:\Users\super\Documents\UniPd\ATA\GutBrainIE\data\GutBrainIE_Full_Collection_2026\Annotations\Dev\json_format\dev.json | exists: True


In [25]:
# --- Train merge with priority ---
train_paths_by_quality = [
    ("gold",   train_files[0]),
    ("silver", train_files[1]),
    ("bronze", train_files[2]),
]

train_data = load_ner_data_priority(train_paths_by_quality)
train_documents = prepare_documents_for_ner(train_data)
print("Total train segments (title+abstract):", len(train_documents))

# --- Optional: add silver_2025 by priority AFTER gold, BEFORE bronze (se vuoi) ---
# Se lo vuoi includere, fai un merge separato:
# train_paths_by_quality = [("gold", ...), ("silver_2025", ...), ("silver", ...), ("bronze", ...)]

# --- Dev ---
if not dev_file.exists():
    raise FileNotFoundError(f"Dev file not found: {dev_file}")

dev_data = load_json(dev_file)
dev_documents = prepare_documents_for_ner(dev_data)
print("Total dev segments (title+abstract):", len(dev_documents))

Loaded 639 docs from train_gold.json (gold)
Loaded 811 docs from train_silver.json (silver)
Loaded 2972 docs from train_bronze.json (bronze)
✓ Merged unique PMIDs: 4422 (priority kept: gold>silver>bronze)
Total train segments (title+abstract): 8844
Total dev segments (title+abstract): 160


In [26]:
# Show example document
example_doc = train_documents[10]
print(f"Example document:")
print(f"  PMID: {example_doc['pmid']}")
print(f"  Location: {example_doc['location']}")
print(f"  Text: {example_doc['text'][:200]}...")
print(f"  Number of entities: {len(example_doc['entities'])}")
print(f"\nFirst 3 entities:")
for entity in example_doc['entities'][:3]:
    print(f"    - '{entity['text_span']}' [{entity['label']}] @ {entity['start_idx']}-{entity['end_idx']}")

Example document:
  PMID: 35833267
  Location: title
  Text: MiR-483-3p improves learning and memory abilities via XPO1 in Alzheimer's disease....
  Number of entities: 3

First 3 entities:
    - 'MiR-483-3p' [chemical] @ 0-9
    - 'XPO1' [chemical] @ 54-57
    - 'Alzheimer's disease' [DDF] @ 62-80


## Initialize BERT Model and Tokenizer
This cell loads
- the BioBERT tokenizer
 - the BioBERT model with a token-classification head sized to our BIO label space

In [27]:
# Initialize tokenizer and model
print("Initializing BERT tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(
    model_name, 
    num_labels=len(label_list), 
    id2label=id2label, 
    label2id=label2id
)

print(f"✓ Tokenizer loaded: {tokenizer.__class__.__name__}")
print(f"✓ Model loaded with {model.num_labels} labels")

# Test tokenization
sample_text = "The gut microbiome plays a role in Parkinson's disease."
tokens = tokenizer.tokenize(sample_text)
print(f"\nSample tokenization: {tokens}")

Initializing BERT tokenizer and model...


C:\Users\super\Documents\UniPd\ATA\GutBrainIE\venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\super\.cache\huggingface\hub\models--microsoft--BiomedNLP-BiomedBERT-base-uncased-abstract. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1226.35it

✓ Tokenizer loaded: BertTokenizer
✓ Model loaded with 27 labels

Sample tokenization: ['the', 'gut', 'microbiome', 'plays', 'a', 'role', 'in', 'parkinson', "'", 's', 'disease', '.']


## Token-label alignment (character spans → BIO token labels)
This cell defines `align_labels_with_tokens`, which converts character-level entity spans into per-token BIO labels. It uses offset mappings from the tokenizer, handles overlaps deterministically (longer span first), prevents double labeling, and converts inclusive dataset end offsets into exclusive offsets for correct matching.

In [28]:
def align_labels_with_tokens(text, entities, tokenizer, label2id, max_length=512):
    encoding = tokenizer(
        text,
        return_offsets_mapping=True,
        return_special_tokens_mask=True,
        add_special_tokens=True,
        truncation=True,
        max_length=max_length,
    )

    input_ids = encoding["input_ids"]
    attention_mask = encoding["attention_mask"]
    offset_mapping = encoding["offset_mapping"]          # (start,end) end exclusive
    special_mask = encoding["special_tokens_mask"]       # 1 if special token

    tokens = tokenizer.convert_ids_to_tokens(input_ids)
    labels = ["O"] * len(input_ids)

    sorted_entities = sorted(
        entities,
        key=lambda e: (int(e["start_idx"]), -(int(e["end_idx"]) - int(e["start_idx"]))),
    )

    labeled_positions = set()

    for ent in sorted_entities:
        ent_start = int(ent["start_idx"])
        ent_end_excl = int(ent["end_idx"]) + 1  # dataset end inclusive -> exclusive
        ent_label = str(ent["label"])

        ent_token_start = None
        ent_token_end = None

        for idx, ((tok_start, tok_end), is_special) in enumerate(zip(offset_mapping, special_mask)):
            if is_special == 1:
                continue
            tok_start = int(tok_start); tok_end = int(tok_end)
            if tok_end <= tok_start:
                continue

            if tok_start < ent_end_excl and tok_end > ent_start:
                if ent_token_start is None:
                    ent_token_start = idx
                ent_token_end = idx

        if ent_token_start is not None and ent_token_end is not None:
            for i in range(ent_token_start, ent_token_end + 1):
                if i in labeled_positions:
                    continue
                tag = f"B-{ent_label}" if i == ent_token_start else f"I-{ent_label}"
                if tag in label2id:
                    labels[i] = tag
                    labeled_positions.add(i)

    label_ids = [label2id.get(tag, label2id["O"]) for tag in labels]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": label_ids,
        "tokens": tokens,
    }

## Apply BIO alignment to the training set
This cell loops over all training segments and converts each one into tokenized inputs (`input_ids`, `attention_mask`) plus aligned BIO labels. The result is stored in `processed_train` for training.

In [29]:
# Process training data
print("Processing training data...")
processed_train = []

for i, doc in enumerate(tqdm(train_documents, desc="Processing train")):
    processed = align_labels_with_tokens(
        doc['text'],
        doc['entities'],
        tokenizer,
        label2id
    )
    processed['pmid'] = doc['pmid']
    processed['location'] = doc['location']
    processed['text'] = doc['text']
    processed['entities'] = doc['entities']
    processed_train.append(processed)

print(f"✓ Training data processed: {len(processed_train)} segments")

Processing training data...


Processing train: 100%|██████████| 8844/8844 [00:07<00:00, 1208.63it/s]

✓ Training data processed: 8844 segments


## Apply BIO alignment to the dev set
This cell repeats the same tokenization + BIO alignment process for the dev segments, producing `processed_dev` for evaluation during training.

In [30]:
# Process dev data
print("Processing dev data...")
processed_dev = []

for i, doc in enumerate(tqdm(dev_documents, desc="Processing dev")):
    processed = align_labels_with_tokens(
        doc['text'],
        doc['entities'],
        tokenizer,
        label2id
    )
    processed['pmid'] = doc['pmid']
    processed['location'] = doc['location']
    processed['text'] = doc['text']
    processed['entities'] = doc['entities']
    processed_dev.append(processed)

print(f"✓ Dev data processed: {len(processed_dev)} segments")

Processing dev data...


Processing dev: 100%|██████████| 160/160 [00:00<00:00, 1092.72it/s]

✓ Dev data processed: 160 segments


#### Inspect aligned token-level labels (debug view)
This cell prints token/label pairs for a sample training segment to visually verify that entity spans were correctly mapped to BIO tags after tokenization.

In [31]:
# Show example with BIO tags
example_idx = 10
example = processed_train[example_idx]

print(f"Example from training data:")
print(f"  Text: {example['text'][:150]}...")
print(f"  Entities: {len(example['entities'])}")
print(f"\nToken-Label pairs (first 30):")

token_label_pairs = []
for token, label_id in zip(example['tokens'][:30], example['labels'][:30]):
    label = id2label[label_id]
    token_label_pairs.append((token, label))

df = pd.DataFrame(token_label_pairs, columns=['Token', 'Label'])
print(df.to_string(index=False))

Example from training data:
  Text: MiR-483-3p improves learning and memory abilities via XPO1 in Alzheimer's disease....
  Entities: 3

Token-Label pairs (first 30):
    Token      Label
    [CLS]          O
      mir B-chemical
        - I-chemical
       48 I-chemical
      ##3 I-chemical
        - I-chemical
       3p I-chemical
 improves          O
 learning          O
      and          O
   memory          O
abilities          O
      via          O
       xp B-chemical
     ##o1 I-chemical
       in          O
alzheimer      B-DDF
        '      I-DDF
        s      I-DDF
  disease      I-DDF
        .          O
    [SEP]          O


## Prepare Dataset for BERT Training

#### Wrap processed examples into a PyTorch Dataset
This cell defines a custom `NERDataset` that returns tensors for `input_ids`, `attention_mask`, and `labels`, making the processed data compatible with the Hugging Face Trainer API.

In [32]:
class NERDataset(Dataset):
    def __init__(self, processed_data):
        self.data = processed_data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "input_ids": torch.tensor(item["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(item["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(item["labels"], dtype=torch.long),
        }

print("✓ Custom dataset class defined")

✓ Custom dataset class defined


#### Build training and evaluation datasets
This cell instantiates `NERDataset` for training and dev splits, and prints dataset sizes as a final sanity check before training.

In [33]:
# Create datasets
print("Creating training datasets...")

train_dataset = NERDataset(processed_train)
dev_dataset = NERDataset(processed_dev)

print(f"✓ Training dataset: {len(train_dataset)} examples")
print(f"✓ Dev dataset: {len(dev_dataset)} examples")

Creating training datasets...
✓ Training dataset: 8844 examples
✓ Dev dataset: 160 examples


## Configure Training Arguments

#### Batch padding for token classification
This cell creates a `DataCollatorForTokenClassification` to pad variable-length sequences within each batch and keep labels aligned with padding (with ignore indices where appropriate). This is required for correct batching in token classification.

In [34]:
# Setup data collator for token classification
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer, padding=True, return_tensors="pt")
print("✓ Data collator initialized")

✓ Data collator initialized


#### Define Evaluation metric (seqeval over BIO tags)
This cell defines `compute_metrics_seqeval` for Hugging Face Trainer. It converts logits to predicted BIO tags (argmax), ignores padding tokens (`-100`), and computes entity-level precision/recall/F1 using seqeval.

In [35]:
import torch
from seqeval.metrics import precision_score, recall_score, f1_score

def compute_metrics_seqeval(p):
    logits, labels = p
    preds = np.argmax(logits, axis=-1)

    true_labels = []
    true_preds = []

    for pred_seq, label_seq in zip(preds, labels):
        seq_true = []
        seq_pred = []
        for p_id, l_id in zip(pred_seq, label_seq):
            if l_id == -100:
                continue
            seq_true.append(id2label[int(l_id)])
            seq_pred.append(id2label[int(p_id)])
        true_labels.append(seq_true)
        true_preds.append(seq_pred)

    return {
        "precision": precision_score(true_labels, true_preds),
        "recall": recall_score(true_labels, true_preds),
        "f1": f1_score(true_labels, true_preds),
    }


###  Training hyperparameters
 This cell sets the key training choices:
 - **learning_rate = 3e-5** with **warmup_ratio=0.1** for stability
 - **gradient_accumulation_steps=2** to increase effective batch size without extra GPU memory
- **num_train_epochs=5** to allow the NER head to converge better than short runs
 - best model selection by F1,
 - fp16 enabled if CUDA is available

In [36]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=output_model_dir,

    # Core optimization
    learning_rate=3e-5,                 # often better than 2e-5 for BioBERT NER
    lr_scheduler_type="linear",
    warmup_ratio=0.1,                   # critical for stability with higher LR
    weight_decay=0.01,

    # Batch/effective batch
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,      # effective batch = 16 (usually helps)

    # Training length
    num_train_epochs=5,                 # 3 is often too short for NER

    # Evaluation / checkpointing
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    label_smoothing_factor=0.0,  # keep 0 for token classification; don't smooth rare labels away
    # If you have compute_metrics with seqeval later, use f1
    metric_for_best_model="f1", #CHANGED
    greater_is_better=True,

    # Runtime / logging
    logging_steps=100,
    save_total_limit=2,
    seed=42,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

print("✓ Training configuration ready")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Learning rate: {training_args.learning_rate}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


✓ Training configuration ready
  Batch size: 8
  Epochs: 5
  Learning rate: 3e-05


## Train BERT Model


### Class-weighted loss (handle label imbalance)
 Many GutBrainIE labels are rare (e.g., some entity types appear much less than "O").
 This cell:
 1) Counts token-level label frequencies in the training set.
 2) Builds class weights using inverse-frequency^power (power=0.5 => sqrt inverse frequency).
 3) Normalizes weights to mean=1 and clips them to avoid extreme gradients

Then it defines a custom Trainer that overrides `compute_loss` to use:
  CrossEntropyLoss(weight=class_weights, ignore_index=-100)
 This generally improves recall for under-represented labels without exploding training.


In [37]:
import torch
from collections import Counter
from transformers import Trainer

def compute_class_weights(processed_train, num_labels, ignore_index=-100, power=0.5):
    """
    Compute class weights from token label counts.
    power=0.5 -> sqrt inverse frequency (usually stable).
    """
    counts = Counter()
    for ex in processed_train:
        for y in ex["labels"]:
            if y == ignore_index:
                continue
            counts[int(y)] += 1

    # build weights: w_c = (1 / freq_c)^power
    freqs = np.zeros(num_labels, dtype=np.float64)
    for c in range(num_labels):
        freqs[c] = counts.get(c, 0)

    # avoid div-by-zero for unseen classes (shouldn't happen, but safe)
    freqs[freqs == 0] = 1.0

    weights = (1.0 / freqs) ** power

    # normalize weights to mean=1 (keeps loss scale reasonable)
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float)

class WeightedLossTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
        logits = outputs.logits  # (B, T, C)

        # flatten
        loss_fct = torch.nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device) if self.class_weights is not None else None,
            ignore_index=-100
        )
        loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))

        return (loss, outputs) if return_outputs else loss


# --- compute weights and inspect FOOD-related weights ---
class_weights = compute_class_weights(processed_train, num_labels=len(label_list), power=0.5)
class_weights = torch.clamp(class_weights, min=0.5, max=5.0) #clipping added

print("Weight(B-food) =", float(class_weights[label2id["B-food"]]))
print("Weight(I-food) =", float(class_weights[label2id["I-food"]]))
print("Weight(O)      =", float(class_weights[label2id["O"]]))
pairs = [(id2label[i], float(class_weights[i])) for i in range(len(label_list))]
pairs_sorted = sorted(pairs, key=lambda x: x[1], reverse=True)
print("Top 10 highest weights:")
for lab, w in pairs_sorted[:10]:
    print(f"{lab:30s} {w:.3f}")



Weight(B-food) = 1.7721912860870361
Weight(I-food) = 1.392630934715271
Weight(O)      = 0.5
Top 10 highest weights:
B-statistical technique        1.979
B-gene                         1.954
B-food                         1.772
B-drug                         1.506
I-food                         1.393
I-gene                         1.328
B-dietary supplement           1.286
B-animal                       1.211
I-statistical technique        1.205
I-drug                         1.192


### Initialize Trainer
This cell creates the custom `WeightedLossTrainer`, wiring together the model, training arguments, datasets, data collator, seqeval metrics, and class weights. This defines the full training + evaluation pipeline.

In [38]:
trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics_seqeval,
    class_weights=class_weights,
)


print("✓ Trainer initialized")
print(f"  Training samples: {len(train_dataset)}")
print(f"  Evaluation samples: {len(dev_dataset)}")

✓ Trainer initialized
  Training samples: 8844
  Evaluation samples: 160


### Run fine-tuning

In [39]:

print("="*60)
print("Starting model training...")
print("="*60)

import time
training_start_time = time.time()

train_result = trainer.train()

training_duration = time.time() - training_start_time

print("\n" + "="*60)
print("✓ TRAINING COMPLETED!")
print("="*60)
print(f"Training time: {training_duration/60:.2f} minutes")

Starting model training...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.458344,0.294939,0.715733,0.850722,0.777411
2,0.370556,0.291520,0.734050,0.849518,0.787574
3,0.282156,0.312584,0.764727,0.838684,0.800000
4,0.236572,0.320347,0.740305,0.857945,0.794796
5,0.194023,0.328589,0.748412,0.851124,0.796470


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.60it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer


✓ TRAINING COMPLETED!
Training time: 9.07 minutes


## Save Trained Model

In [40]:
# Save the trained model
print("Saving trained model...")

os.makedirs(output_model_dir, exist_ok=True)
trainer.save_model(output_model_dir)
tokenizer.save_pretrained(output_model_dir)

print(f"✓ Model saved to: {output_model_dir}")

Saving trained model...


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.97it/s]

✓ Model saved to: models/pubmedbert_ner_twopass_gold_silver_bronze
